# 01 license lint

Question: does the license lint of Golem, `.github/scripts/license_lint.py`, pass the tree of this
revision and refuse every break of its nine rules, and does it agree with `reuse lint`, the tool of
the REUSE specification, wherever both judge the same thing.

Documentation/licensing.md states the rules, and the docstring of the script numbers them. The
notebook builds a small tree that follows every rule, breaks one rule at a time in a copy of it,
and runs both tools on each copy. A case names the rule the lint must report, or none when the copy
must pass, and what `reuse lint` must say, or nothing where REUSE does not judge that rule, such as
the place of the SPDX line, which REUSE leaves open.

In [ ]:
# Papermill overrides this cell.
# The tree under test, the root of the repository.
golem_root = "."

# The holder every file of Golem names.
holder = "Florian Kleber <opensource@skytekx.com>"

## Setup

The cell finds the repository, prints the versions of Python and reuse, and defines the minimal
tree: `LICENSE`, its copy `LICENSES/AGPL-3.0-only.txt`, and one Rust file with both SPDX lines.

In [ ]:
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "tests" / "harness" / "__init__.py").is_file():
        sys.path.insert(0, str(_base / "tests"))
        break

from harness import paths, run

P = paths(golem_root)
LINT = P.root / ".github" / "scripts" / "license_lint.py"
REUSE = [sys.executable, "-m", "reuse"]

print("python:", sys.version.split()[0])
print("reuse:", run(REUSE + ["--version"], check=True).stdout.splitlines()[0])
print("lint:", P.rel(LINT))

LICENSE_BYTES = (P.root / "LICENSE").read_bytes()
MIT_BYTES = (P.root / "LICENSES" / "MIT.txt").read_bytes()
HEADER = "// SPDX-License-Identifier: AGPL-3.0-only\n// SPDX-FileCopyrightText: 2026 %s\n" % holder
SCRATCH = Path(tempfile.mkdtemp(prefix="golem-01-"))


def base_tree(name):
    # A tree that follows every rule, in a directory of its own.
    tree = SCRATCH / name
    (tree / "LICENSES").mkdir(parents=True)
    (tree / "LICENSE").write_bytes(LICENSE_BYTES)
    (tree / "LICENSES" / "AGPL-3.0-only.txt").write_bytes(LICENSE_BYTES)
    write(tree, "src/main.rs", HEADER + "fn main() {}\n")
    return tree


def write(tree, relative, text):
    path = tree / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    if isinstance(text, bytes):
        path.write_bytes(text)
    else:
        path.write_text(text, encoding="utf-8")


def judge(tree):
    # Both tools on one tree: the exit status and output of the lint, and whether REUSE passes.
    lint = run(["python3", "-I", LINT, "--root", tree])
    reuse = run(REUSE + ["--root", tree, "lint", "--quiet"])
    return lint, reuse.returncode == 0

## The tree of this revision

Both tools must pass the repository itself, so that CI, which runs the same two commands, is green
on this revision.

In [ ]:
lint = run(["python3", "-I", LINT], cwd=P.root)
print(lint.stdout.strip())
assert lint.returncode == 0, "the license lint fails the tree of this revision:\n" + lint.stdout
reuse = run(REUSE + ["lint"], cwd=P.root)
print(reuse.stdout.strip().splitlines()[-1])
assert reuse.returncode == 0, "reuse lint fails the tree of this revision:\n" + reuse.stdout

## The ports of this tree

`ports.toml` names the upstream of every port as the plugin and its version, as NOTICE names it,
and the source of exactly that version. The cell holds each port to the same upstream in three
places: the constant `UPSTREAM` of the crate where the crate has one, the section of NOTICE headed
by the name of the plugin, whose `Version:` line starts with the version, and the source, which
names the version as its tag or the commit of that `Version:` line.

In [ ]:
import re
import tomllib

PORTS = tomllib.loads((P.root / "ports.toml").read_text(encoding="utf-8"))["port"]
NOTICE_LINES = (P.root / "NOTICE").read_text(encoding="utf-8").splitlines()


def notice_section(project):
    # The lines of the section of NOTICE whose heading, underlined with =, is the project.
    def heading(index):
        return index + 1 < len(NOTICE_LINES) and NOTICE_LINES[index] and NOTICE_LINES[index + 1] == "=" * len(NOTICE_LINES[index])

    for index, line in enumerate(NOTICE_LINES):
        if line == project and heading(index):
            end = next((other for other in range(index + 2, len(NOTICE_LINES)) if heading(other)), len(NOTICE_LINES))
            return NOTICE_LINES[index + 2:end]
    return None


wrong = []
for port in PORTS:
    project, _, version = port["upstream"].rpartition(" ")
    if not project or not re.fullmatch(r"[0-9][0-9A-Za-z.+-]*", version):
        wrong.append("%s: the upstream %r names no version" % (port["name"], port["upstream"]))
        continue
    lib = P.root / port["path"] / "src" / "lib.rs"
    constant = re.search(r'pub const UPSTREAM: &str = "([^"]*)";', lib.read_text(encoding="utf-8")) if lib.is_file() else None
    if constant and constant.group(1) != port["upstream"]:
        wrong.append("%s: the crate names its upstream %r, ports.toml %r" % (port["name"], constant.group(1), port["upstream"]))
    section = notice_section(project)
    versions = [line[len("Version: "):] for line in section or [] if line.startswith("Version: ")]
    if section is None:
        wrong.append("%s: NOTICE has no section headed %s" % (port["name"], project))
    elif len(versions) != 1 or not re.match(re.escape(version) + r"(,|$)", versions[0]):
        wrong.append("%s: the section %s of NOTICE gives the version %s, not %s" % (port["name"], project, versions, version))
    commit = re.search(r"\bcommit ([0-9a-f]{40})\b", versions[0]) if len(versions) == 1 else None
    pinned = re.search(r"/tree/([^/]+)", port["source"])
    if not pinned or pinned.group(1) not in {version, commit.group(1) if commit else version}:
        wrong.append("%s: the source %s pins neither the version %s nor the commit of NOTICE" % (port["name"], port["source"], version))
    print("%-24s %-26s crate %-26s source at %s" % (port["name"], port["upstream"], constant.group(1) if constant else "-",
                                                  pinned.group(1)[:12] if pinned else "nothing"))
assert not wrong, "\n".join(wrong)

## The cases

Each case is a name, a change to the minimal tree, the rule the lint must report or `None` when
the tree must pass, and what `reuse lint` must say: `True` for pass, `False` for fail, `None` where
REUSE does not judge that rule. A case that reports a rule must report it for the path the case
broke.

In [ ]:
def mutate_license(tree):
    write(tree, "LICENSE", LICENSE_BYTES.replace(b"19 November 2007", b"19 November 2008"))


def drop_text(tree):
    (tree / "LICENSES" / "AGPL-3.0-only.txt").unlink()


def tagged(expression, holder_line=None):
    holder_line = holder if holder_line is None else holder_line
    return "// SPDX-License-Identifier: %s\n// SPDX-FileCopyrightText: 2026 %s\n" % (expression, holder_line)


def workspace(tree):
    write(tree, "Cargo.toml", "# SPDX-License-Identifier: AGPL-3.0-only\n# SPDX-FileCopyrightText: 2026 %s\n"
          "[workspace]\nmembers = [\"crate\"]\n[workspace.package]\nlicense = \"AGPL-3.0-only\"\n" % holder)
    write(tree, "crate/Cargo.toml", "# SPDX-License-Identifier: AGPL-3.0-only\n# SPDX-FileCopyrightText: 2026 %s\n"
          "[package]\nname = \"crate\"\nlicense.workspace = true\n" % holder)


def with_mit(tree, notice):
    write(tree, "LICENSES/MIT.txt", MIT_BYTES)
    write(tree, "src/derived.rs", tagged("AGPL-3.0-only AND MIT") + "fn derived() {}\n")
    if notice:
        write(tree, "NOTICE", "Golem\n\nAosabi, MIT\n")
        write(tree, "REUSE.toml", "# SPDX-License-Identifier: AGPL-3.0-only\n# SPDX-FileCopyrightText: 2026 %s\n"
              "version = 1\n[[annotations]]\npath = \"NOTICE\"\nSPDX-FileCopyrightText = \"2026 %s\"\n"
              "SPDX-License-Identifier = \"AGPL-3.0-only\"\n" % (holder, holder))


PORT_TABLE = """# SPDX-License-Identifier: AGPL-3.0-only
# SPDX-FileCopyrightText: 2026 %s
[[port]]
name = "golem-permissions"
path = "crates/golem-permissions"
upstream = "LuckPerms 5.5.71"
source = "https://github.com/LuckPerms/LuckPerms"
%s
holders = ["lucko (Luck) <luck@lucko.me>", "LuckPerms contributors"]
"""


def ported(expression="MIT", holders=("lucko (Luck) <luck@lucko.me>", "LuckPerms contributors"),
           manifest='license = "MIT"', notice="LuckPerms 5.5.71", table='license = "MIT"'):
    # A port of LuckPerms as ports.toml names it, with one of its parts changed.
    def change(tree):
        write(tree, "LICENSES/MIT.txt", MIT_BYTES)
        write(tree, "ports.toml", PORT_TABLE % (holder, table))
        tags = ["SPDX-License-Identifier: %s" % expression, "SPDX-FileCopyrightText: 2026 %s" % holder]
        tags += ["SPDX-FileCopyrightText: %s" % name for name in holders]
        write(tree, "crates/golem-permissions/src/lib.rs", "".join("// %s\n" % tag for tag in tags) + "//! A port.\n")
        write(tree, "crates/golem-permissions/Cargo.toml", "".join("# %s\n" % tag for tag in tags)
              + "[package]\nname = \"golem-permissions\"\n%s\n" % manifest)
        write(tree, "NOTICE", "Golem\n\n%s, MIT\n" % notice)
        write(tree, "REUSE.toml", "# SPDX-License-Identifier: AGPL-3.0-only\n# SPDX-FileCopyrightText: 2026 %s\n"
              "version = 1\n[[annotations]]\npath = \"NOTICE\"\nSPDX-FileCopyrightText = \"2026 %s\"\n"
              "SPDX-License-Identifier = \"AGPL-3.0-only\"\n" % (holder, holder))
    return change


CASES = [
    # name, change, rule and path the lint reports or None, what reuse lint says
    ("the minimal tree", lambda t: None, None, True),
    ("1: LICENSE changed by one byte", mutate_license, (1, "LICENSE"), None),
    ("1: the text of the AGPL missing", drop_text, (1, "LICENSES/AGPL-3.0-only.txt"), False),
    ("2: an operator in lower case", lambda t: write(t, "src/a.rs", tagged("AGPL-3.0-only and MIT")), (2, "src/a.rs"), None),
    ("2: a broken expression", lambda t: write(t, "src/a.rs", tagged("AGPL-3.0-only AND")), (2, "src/a.rs"), False),
    ("3: a license without its text", lambda t: write(t, "src/a.rs", tagged("GPL-2.0-only")), (3, "src/a.rs"), False),
    ("3: AGPL-3.0-or-later, which Golem never takes", lambda t: write(t, "src/a.rs", tagged("AGPL-3.0-or-later")), (3, "src/a.rs"), False),
    ("3: a text without .txt", lambda t: write(t, "LICENSES/AGPL-3.0-only", LICENSE_BYTES), (3, "LICENSES/AGPL-3.0-only"), False),
    ("4: a crate without a license", lambda t: write(t, "Cargo.toml", "# SPDX-License-Identifier: AGPL-3.0-only\n"
        "# SPDX-FileCopyrightText: 2026 %s\n[package]\nname = \"golem\"\n" % holder), (4, "Cargo.toml"), True),
    ("4: a crate that inherits the workspace license", workspace, None, True),
    ("5: a fork under third_party/ without its license file", lambda t: write(t, "third_party/fork/Cargo.toml",
        "[package]\nname = \"fork\"\nlicense = \"MIT\"\n"), (5, "third_party/fork"), None),
    ("6: a file without tags", lambda t: write(t, "src/a.rs", "fn a() {}\n"), (6, "src/a.rs"), False),
    ("6: the tag on the third line", lambda t: write(t, "src/a.rs", "// a\n\n" + tagged("AGPL-3.0-only")), (6, "src/a.rs"), True),
    ("6: the tag under a #! line", lambda t: write(t, "tools/a.py", "#!/usr/bin/env python3\n# SPDX-License-Identifier: AGPL-3.0-only\n"
        "# SPDX-FileCopyrightText: 2026 %s\n" % holder), None, True),
    ("6: the tags of a wiki page in its front matter", lambda t: write(t, "okf/index.md", "---\n# SPDX-License-Identifier: AGPL-3.0-only\n"
        "# SPDX-FileCopyrightText: 2026 %s\ntype: Index\n---\n# Golem\n" % holder), None, True),
    ("6: a wiki page with a license key and no tag", lambda t: write(t, "okf/index.md", "---\nlicense: AGPL-3.0-only\n---\n"), (6, "okf/index.md"), False),
    ("6: JSON with a .license file beside it", lambda t: (write(t, "config/a.json", "{}\n"), write(t, "config/a.json.license",
        "SPDX-License-Identifier: AGPL-3.0-only\nSPDX-FileCopyrightText: 2026 %s\n" % holder)), None, True),
    ("6: JSON without a .license file", lambda t: write(t, "config/a.json", "{}\n"), (6, "config/a.json"), False),
    ("7: a file under MIT without NOTICE", lambda t: with_mit(t, notice=False), (7, "NOTICE"), True),
    ("7: a file under MIT with NOTICE", lambda t: with_mit(t, notice=True), None, True),
    ("8: a file without a copyright line", lambda t: write(t, "src/a.rs", "// SPDX-License-Identifier: AGPL-3.0-only\n"), (8, "src/a.rs"), False),
    ("8: a copyright line of another holder alone", lambda t: write(t, "src/a.rs", tagged("AGPL-3.0-only", "Someone Else")), (8, "src/a.rs"), True),
    ("9: a port with the license and the holders of its upstream", ported(), None, True),
    ("9: a file of a port under the AGPL", ported(expression="AGPL-3.0-only"), (9, "crates/golem-permissions/src/lib.rs"), None),
    ("9: a file of a port without the holders of its upstream", ported(holders=()), (9, "crates/golem-permissions/src/lib.rs"), True),
    ("9: a port that takes the license of the workspace", ported(manifest="license.workspace = true"), (9, "crates/golem-permissions/Cargo.toml"), True),
    ("9: NOTICE without the upstream of a port", ported(notice="Aosabi 0.1.0"), (9, "NOTICE"), True),
    ("9: a port of ports.toml without its license", ported(table=""), (9, "ports.toml port 1"), True),
]
print("cases:", len(CASES))

In [ ]:
rows, failures = [], []
for index, (name, change, expected, reuse_expected) in enumerate(CASES):
    tree = base_tree("case-%02d" % index)
    change(tree)
    lint, reuse_passed = judge(tree)
    lines = [line for line in lint.stdout.splitlines() if ": rule " in line]
    if expected is None:
        lint_ok = lint.returncode == 0 and not lines
    else:
        rule, path = expected
        lint_ok = lint.returncode == 1 and any(line.startswith("license_lint: %s: rule %d:" % (path, rule)) for line in lines)
    reuse_ok = reuse_expected is None or reuse_passed == reuse_expected
    rows.append((name, "pass" if expected is None else "rule %d" % expected[0], "ok" if lint_ok else "WRONG",
                 {True: "pass", False: "fail", None: "-"}[reuse_expected], "pass" if reuse_passed else "fail",
                 "ok" if reuse_ok else "WRONG"))
    if not (lint_ok and reuse_ok):
        failures.append((name, lint.stdout, reuse_passed))

width = max(len(row[0]) for row in rows)
print("%-*s  %-7s %-6s  %-6s %-6s %s" % (width, "case", "lint", "", "reuse", "got", ""))
for row in rows:
    print("%-*s  %-7s %-6s  %-6s %-6s %s" % (width, *row))
for name, output, reuse_passed in failures:
    print("\n--- %s, reuse %s\n%s" % (name, "passed" if reuse_passed else "failed", output))
assert not failures, "%d of %d cases went wrong" % (len(failures), len(CASES))

## Git work trees

In a git work tree the lint reads the files git tracks and nothing else, so an untracked file
without tags, such as the output of a run, does not fail it, while the same file fails once it is
added.

In [ ]:
tree = base_tree("git")
run(["git", "init", "-q", tree], check=True)
run(["git", "-C", tree, "add", "-A"], check=True)
write(tree, "scratch.txt", "untracked output\n")
untracked = run(["python3", "-I", LINT, "--root", tree])
print("untracked:", untracked.stdout.strip().splitlines()[-1])
assert untracked.returncode == 0, untracked.stdout
run(["git", "-C", tree, "add", "scratch.txt"], check=True)
tracked = run(["python3", "-I", LINT, "--root", tree])
print("tracked:  ", tracked.stdout.strip().splitlines()[-1])
assert tracked.returncode == 1 and "scratch.txt: rule 6:" in tracked.stdout, tracked.stdout

## Verdict

The cell clears the scratch trees and prints the count. A run that reaches it passed every case.

In [ ]:
shutil.rmtree(SCRATCH)
print("verdict: the license lint passes the tree and judges %d cases as expected, in agreement with reuse lint" % len(CASES))